# get-children-callable-param — worked example 1: get_children scans __dict__ for tensor attributes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `get-children-callable-param`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A mini `nn.Module` exposes its trainable state by scanning `self.__dict__` and yielding `(name, value)` for every attribute that is a `MiniTensor`. Configuration attributes (ints, strings) are skipped via an `isinstance` check. Yielding (rather than returning a list) keeps the walk lazy and composable.

## Worked solution

We implement `get_children` as a generator over a module's tensor-valued attributes.

1. Instance attributes live in `self.__dict__`, an insertion-ordered dict of `{name: value}`.
2. We iterate `self.__dict__.items()` so the yield order matches definition order — important for stable state-dict naming downstream.
3. For each `(name, val)` we test `isinstance(val, MiniTensor)`. Only tensor-valued attributes are trainable state; an `int` like `in_features` is config and must not appear.
4. We `yield name, val` for the survivors. The caller consumes it with a `for` loop, paying zero list-allocation cost.
5. We build a `Linear` with two tensors plus two config ints and confirm only the two tensors come back.

In [ ]:
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, data, requires_grad=False):
        self.data = data
        self.requires_grad = requires_grad

class Module:
    def get_children(self):
        for name, val in self.__dict__.items():
            if isinstance(val, MiniTensor):
                yield name, val

class Linear(Module):
    def __init__(self):
        self.weight = MiniTensor(t.randn(4, 3), requires_grad=True)
        self.bias = MiniTensor(t.zeros(4), requires_grad=True)
        self.in_features = 3
        self.out_features = 4

names = [name for name, _ in Linear().get_children()]
print(names)
print('only tensors:', names == ['weight', 'bias'])